# ITensors Benchmark

Beware that this notebook takes a few hours to run due to the long-time integration of the diffusion PDE.

Running this notebook will generate a CSV file with the times and accuracies achieved. To plot the results, you must run `03_plot_benchmarks.ipynb` afterwards.

In [1]:
# Configuration
const N_REPS     = 101
const N_VALUES   = 4:14
const TOLERANCE  = 1e-6
const OUTPUT_CSV = "results_itensors.csv"

"results_itensors.csv"

In [2]:
## Uncomment to install dependencies

# using Pkg
# Pkg.add(["CSV", "DataFrames", "ITensors", "ITensorMPS", "KrylovKit"])

In [3]:
using CSV, DataFrames
using ITensors, ITensorMPS, KrylovKit
using LinearAlgebra, Printf, Random, SparseArrays, Statistics

# Disable warning on exceeding 14 indices
ITensors.disable_warn_order()

# Set number of threads
BLAS.set_num_threads(parse(Int, get(ENV, "BENCH_THREADS", "1")))
Random.seed!(0)

TaskLocalRNG()

In [4]:
# Truncation tolerance
bond_cutoff(nsites::Int) = TOLERANCE^2 / max(nsites - 1, 1)

# Periodic tridiagonal MPO
function periodic_tridiag_mpo(sites, a::Real, b::Real, c::Real)
    N = length(sites)
    links = [Index(3, "Link,l=$n") for n in 1:(N - 1)]
    t = Vector{ITensor}(undef, N)

    A = zeros(3, 2, 2, 3)
    A[1, 1, 1, 1] = 1.0;  A[1, 2, 2, 1] = 1.0
    A[2, 2, 1, 1] = 1.0;  A[2, 1, 2, 2] = 1.0
    A[3, 1, 2, 1] = 1.0;  A[3, 2, 1, 3] = 1.0

    Larr = a * A[1, :, :, :] + b * A[2, :, :, :] + c * A[3, :, :, :]
    t[1] = itensor(Larr, sites[1]', dag(sites[1]), links[1])

    for n in 2:(N - 1)
        t[n] = itensor(copy(A), links[n - 1], sites[n]', dag(sites[n]), links[n])
    end

    Rarr = A[:, :, :, 1] + A[:, :, :, 2] + A[:, :, :, 3]
    t[N] = itensor(Rarr, links[N - 1], sites[N]', dag(sites[N]))

    return MPO(t)
end

# Laplacian via periodic finite differences
function laplacian_mpo(sites, dx::Real, nu::Real)
    c = nu / dx^2
    return periodic_tridiag_mpo(sites, -2c, c, c)
end

# Transverse-field Ising model: H = -Σ Sz·Sz - Σ Sx
function tfim_hamiltonian(sites)
    os = OpSum()
    for j in 1:(length(sites) - 1)
        os += -1.0, "Sz", j, "Sz", j + 1
    end
    for j in 1:length(sites)
        os += -1.0, "Sx", j
    end
    return MPO(os, sites)
end

# Dense Hamiltonian matrix for exact reference solutions
function dense_hamiltonian(sites)
    H = contract(tfim_hamiltonian(sites))
    return matrix(combiner(prime.(sites)...) * H * combiner(sites...))
end

# Reference ground-state energy
function reference_ground_energy(N::Int)
    sites = siteinds("S=1/2", N)
    Hmat = dense_hamiltonian(sites)
    vals, _ = eigsolve(Hmat, ones(2^N), 1, :SR; tol = 1e-12, maxiter = 1000)
    return real(vals[1])
end

# MPS 
state_vector(psi::MPS, sites) =
    Vector{ComplexF64}(array(combiner(sites...) * contract(psi)))

contract_to_vector(psi::MPS, nx::Int) =
    Vector{Float64}(reshape(Array(contract(psi).tensor), nx))

# Single RK4 step
function rk4_step(L::MPO, u::MPS, dt::Real; cutoff::Real)
    k1 = apply(L, u; cutoff, normalize = false)
    k2 = apply(L, +(u, (0.5dt) * k1; cutoff); cutoff, normalize = false)
    k3 = apply(L, +(u, (0.5dt) * k2; cutoff); cutoff, normalize = false)
    k4 = apply(L, +(u, dt * k3; cutoff); cutoff, normalize = false)
    return +(u, (dt/6)*k1, (dt/3)*k2, (dt/3)*k3, (dt/6)*k4; cutoff)
end

rk4_step (generic function with 1 method)

In [5]:
# Ground-state energy via DMRG. Accuracy: relative error vs. exact diagonalization.
function benchmark_dmrg(N::Int; nreps::Int = N_REPS)
    sites = siteinds("S=1/2", N)
    H = tfim_hamiltonian(sites)
    E0 = reference_ground_energy(N)

    # Initial state
    Random.seed!(0)
    psi0 = normalize(random_mps(sites; linkdims = 4))

    # Run benchmark
    times, accs = Float64[], Float64[]
    energy, psi = NaN, psi0
    for _ in 1:nreps
        observer = DMRGObserver(; energy_tol = TOLERANCE)
        elapsed = @elapsed energy, psi = dmrg(
            H, psi0; nsweeps = 20, maxdim = typemax(Int),
            cutoff = bond_cutoff(N), observer, outputlevel = 0,
        )
        push!(times, elapsed)
        push!(accs, abs(energy - E0) / max(abs(E0), 1.0))
    end

    return (problem = "hamiltonian", subproblem = "dmrg", library = "itensors",
            N = N, D = maxlinkdim(psi), time_samples = join(times, ";"),
            accuracy_samples = join(accs, ";"))
end


# Time evolution via 2-site TDVP
# Backend "applyexp" for faster execution
function benchmark_tdvp(N::Int; nreps::Int = N_REPS)
    sites = siteinds("S=1/2", N)
    H = tfim_hamiltonian(sites)
    psi0 = MPS(sites, ["X+" for _ in 1:N])
    T, steps = 0.05, 50

    # Reference solution
    Hmat = dense_hamiltonian(sites)
    v0 = state_vector(psi0, sites)
    exact, _ = exponentiate(Hmat, -im * T, v0; tol = 1e-12)
    exact_norm = norm(exact)

    # Run benchmark
    times, accs = Float64[], Float64[]
    psi_T = psi0
    for _ in 1:nreps
        elapsed = @elapsed psi_T = tdvp(
            H, -im * T, psi0;
            time_step = -im * T / steps, nsteps = steps,
            maxdim = typemax(Int), cutoff = bond_cutoff(N),
            normalize = false, outputlevel = 0, nsite = 2,
            updater_backend = "applyexp",
        )
        push!(times, elapsed)
        push!(accs, norm(state_vector(psi_T, sites) - exact) / exact_norm)
    end

    return (problem = "hamiltonian", subproblem = "tdvp", library = "itensors",
            N = N, D = maxlinkdim(psi_T), time_samples = join(times, ";"),
            accuracy_samples = join(accs, ";"))
end


# Diffusion PDE via RK4
function benchmark_pde(N_input::Int; nreps::Int = N_REPS)
    Nq = max(N_input, 4)
    sites = siteinds("Qubit", Nq)
    nx, dx = 2^Nq, 2.0 / 2^Nq
    nu, sigma, x0 = 0.01, 0.1, 1.0

    # Space and initial conditions
    x = range(0.0, 2.0; length = nx + 1)[1:(end - 1)]
    psi0_vec = exp.(-(x .- x0) .^ 2 / (2sigma^2))
    L = laplacian_mpo(sites, dx, nu)
    psi0 = MPS(psi0_vec, sites; cutoff = bond_cutoff(Nq))

    steps = 1000
    dt = 0.1 * (0.5dx^2 / nu) # From stability condition
    T = steps * dt

    # Analytic solution
    sigma_t = sqrt(sigma^2 + 2nu * T)
    psi_exact = exp.(-(x .- x0) .^ 2 / (2sigma_t^2)) .* (sigma / sigma_t)

    # Run benchmark
    times, errs = Float64[], Float64[]
    psi = psi0
    for _ in 1:nreps
        psi = psi0
        elapsed = @elapsed for _ in 1:steps
            psi = rk4_step(L, psi, dt; cutoff = bond_cutoff(Nq))
        end

        push!(times, elapsed)
        v = contract_to_vector(psi, nx)
        push!(errs, norm(v - psi_exact) / norm(psi_exact))
    end

    return (problem = "pde", subproblem = "diffusion_rk4", library = "itensors",
            N = Nq, D = maxlinkdim(psi), time_samples = join(times, ";"),
            accuracy_samples = join(errs, ";"))
end

benchmark_pde (generic function with 1 method)

In [6]:
run_benchmarks(N; nreps = N_REPS) =
    [benchmark_dmrg(N; nreps), benchmark_tdvp(N; nreps), benchmark_pde(N; nreps)]

# Warmup
run_benchmarks(first(N_VALUES); nreps = 1)

# Run benchmark
records = []
for N in N_VALUES, r in run_benchmarks(N)
    tmed = median(parse.(Float64, split(r.time_samples, ";")))
    emed = median(parse.(Float64, split(r.accuracy_samples, ";")))
    @printf("%-14s N=%2d D=%2d  t=%.4fs  err=%.3e\n", r.subproblem, r.N, r.D, tmed, emed)
    push!(records, r)
end

# Save results
CSV.write(OUTPUT_CSV, DataFrame(records))
println("Wrote $(length(records)) records to $OUTPUT_CSV")

dmrg           N= 4 D= 4  t=0.0030s  err=4.241e-16
tdvp           N= 4 D= 2  t=0.0580s  err=4.340e-10
diffusion_rk4  N= 4 D= 3  t=4.0914s  err=7.434e-01
dmrg           N= 5 D= 4  t=0.0041s  err=5.073e-16
tdvp           N= 5 D= 2  t=0.0823s  err=6.137e-10
diffusion_rk4  N= 5 D= 4  t=5.4654s  err=1.619e-01
dmrg           N= 6 D= 6  t=0.0055s  err=1.688e-15
tdvp           N= 6 D= 2  t=0.1067s  err=7.516e-10
diffusion_rk4  N= 6 D= 6  t=7.1103s  err=4.237e-03
dmrg           N= 7 D= 6  t=0.0070s  err=4.815e-15
tdvp           N= 7 D= 2  t=0.1322s  err=8.679e-10
diffusion_rk4  N= 7 D= 7  t=8.6940s  err=5.464e-04
dmrg           N= 8 D= 6  t=0.0085s  err=2.378e-14
tdvp           N= 8 D= 2  t=0.1559s  err=9.704e-10
diffusion_rk4  N= 8 D= 7  t=10.5469s  err=1.776e-04
dmrg           N= 9 D= 6  t=0.0103s  err=7.400e-14
tdvp           N= 9 D= 2  t=0.1782s  err=1.063e-09
diffusion_rk4  N= 9 D= 7  t=12.8993s  err=3.594e-04
dmrg           N=10 D= 7  t=0.0131s  err=9.916e-14
tdvp           N=10 D= 2  t=0